# Populate Tier-1 feature cache (ESM-2, AlphaFold-DB, MACE-OFF)

**This notebook runs on Colab Pro with a GPU** (A100 preferred,
T4 fallback). It populates the three Tier-1 parquet files under
`cell_sim/features/cache/`:

| Extractor        | Source                                  | Wall time (A100) | Parquet size |
|------------------|-----------------------------------------|------------------|--------------|
| ESM-2 (650M)     | `facebook/esm2_t33_650M_UR50D`          | ~2 min           | ~2.3 MB      |
| AlphaFold-DB     | `https://alphafold.ebi.ac.uk/files/AF-*` | ~15–30 min (net) | ~100 KB      |
| MACE-OFF (small) | `mace-off23_small`                      | ~5 min           | ~500 KB      |

After running all cells, one of three output modes writes the
parquets + refreshed manifest back to your machine / Drive /
repo. Choose in cell 9.

**Non-goals**: this notebook does **not** run any detector, does
**not** measure MCC, and does **not** touch
`cell_sim/layer6_essentiality/`. That is Session 15's job.


In [ ]:
# Cell 2 — install the GPU-side deps. These are intentionally NOT in
# cell_sim/requirements.txt — see the comment there.
#
# The install is split into two groups:
#   1. Core stack (ESM-2 + AlphaFold + framework). Required.
#   2. MACE-OFF stack. Optional — mace-torch pins an older e3nn, so
#      we let pip resolve mace's own compatible version and tolerate
#      a failure (ESM-2 + AlphaFold still populate).

# --- Group 1: core stack (ESM-2, AlphaFold, supporting libs) ---
!pip install -q \
    "torch>=2.2" \
    "transformers>=4.40" \
    "biopython>=1.83" \
    "rdkit>=2023.9" \
    "ase>=3.22" \
    "pyarrow>=14" \
    "pandas>=2" \
    "numpy>=1.24"

# --- Group 2: MACE-OFF (optional, may fail; ESM-2 / AlphaFold survive) ---
# NOTE: do NOT pin e3nn here — mace-torch's own metadata selects a
# compatible version. Pinning e3nn>=0.5.1 breaks resolution on
# every mace-torch 0.3.x release (they pin e3nn<0.5).
import subprocess, sys
_mace_status = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mace-torch"],
    capture_output=True, text=True,
)
MACE_AVAILABLE = (_mace_status.returncode == 0)
if not MACE_AVAILABLE:
    print("[warn] mace-torch install failed; MACE-OFF cell will be skipped.")
    print("       last pip stderr line:")
    for line in _mace_status.stderr.strip().splitlines()[-3:]:
        print(f"         {line}")
else:
    print("mace-torch install OK")

import torch
print(f"\ntorch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  VRAM: {props.total_memory / 1024**3:.1f} GiB")
else:
    print("  CPU only — ESM-2 will run but will take much longer.")
print(f"MACE_AVAILABLE = {MACE_AVAILABLE}")


In [ ]:
# Cell 3 — clone the project repo at the current Session-13 HEAD.
# If you've already cloned into /content/cell, the second command
# just switches into it without re-cloning.
import os
if not os.path.isdir("/content/cell"):
    !git clone --branch claude/syn3a-whole-cell-simulator-REjHC \
        https://github.com/Nikku03/cell.git /content/cell
%cd /content/cell

import sys
sys.path.insert(0, "/content/cell")
print("CWD:", os.getcwd())
!git log --oneline -1


In [ ]:
# Cell 4 — load Syn3A CDS sequences from the staged GenBank file.
# We parse the .gb directly rather than via cell_sim.layer0_genome
# because we want the translated protein sequences (stored as
# /translation qualifiers on CDS features in CP016816.2).
from pathlib import Path
from Bio import SeqIO
import pandas as pd

GB_PATH = Path(
    "cell_sim/data/Minimal_Cell_ComplexFormation/input_data/syn3A.gb"
)
records = list(SeqIO.parse(GB_PATH, "genbank"))
assert len(records) == 1, f"expected 1 record, got {len(records)}"
rec = records[0]
print(f"accession={rec.id}  length={len(rec.seq):,} bp")

cds_rows = []
for feat in rec.features:
    if feat.type != "CDS":
        continue
    locus = (feat.qualifiers.get("locus_tag") or [""])[0]
    gene_name = (feat.qualifiers.get("gene") or [""])[0]
    translation = (feat.qualifiers.get("translation") or [""])[0]
    protein_id = (feat.qualifiers.get("protein_id") or [""])[0]
    product = (feat.qualifiers.get("product") or [""])[0]
    if not locus or not translation:
        continue
    cds_rows.append({
        "locus_tag": locus,
        "gene_name": gene_name,
        "protein_id": protein_id,
        "product": product,
        "sequence": translation.upper(),
    })
cds_df = pd.DataFrame(cds_rows)
print(f"\n{len(cds_df)} CDS with translations")
print(cds_df.head(3))
dnaA_seq = cds_df.loc[cds_df["gene_name"] == "dnaA", "sequence"].iloc[0]
print(f"\ndnaA length: {len(dnaA_seq)} aa; starts: {dnaA_seq[:10]}")


In [ ]:
# Cell 5 — ESM-2 650M embeddings for every CDS with a translation.
import hashlib, time
from pathlib import Path
from cell_sim.features.extractors import ESM2Extractor
from cell_sim.features.batched_inference import (
    BatchedInferenceConfig,
)

CACHE_DIR = Path("cell_sim/features/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

esm = ESM2Extractor()
cfg = BatchedInferenceConfig(
    batch_size=16,
    device="auto",
    dtype="float16" if torch.cuda.is_available() else "float32",
    max_seq_length=1022,
    progress=True,
)

t0 = time.perf_counter()
esm_feats = esm.extract(cds_df[["locus_tag", "sequence"]], cfg)
esm_wall = time.perf_counter() - t0
esm_path, esm_sha = esm.write_cache(esm_feats, cache_dir=CACHE_DIR)
print(f"ESM-2 done in {esm_wall:.1f}s  parquet={esm_path}")
print(f"  rows={len(esm_feats)}  dims={len(esm.feature_cols)}")
print(f"  sha256={esm_sha}")
print("\nfirst 5 locus tags' embedding L2 norms:")
for locus in esm_feats.index[:5]:
    vec = esm_feats.loc[locus]
    print(f"  {locus}  ||x||={(vec ** 2).sum() ** 0.5:.3f}")


In [ ]:
# Cell 6 — AlphaFold-DB per-protein structural descriptors.
# UniProt IDs are looked up in the existing Syn3A proteome table
# via the GenBank xref; missing IDs (some tRNA / ncRNA CDS overlaps
# lack a UniProt) become no-structure rows (NaN features, has_structure=0).
import re
from cell_sim.features.extractors import AlphaFoldExtractor

# Build the uniprot_id lookup from GenBank CDS xref qualifiers.
uniprot_map = {}
for feat in rec.features:
    if feat.type != "CDS":
        continue
    locus = (feat.qualifiers.get("locus_tag") or [""])[0]
    if not locus:
        continue
    for x in feat.qualifiers.get("db_xref", []):
        m = re.match(r"UniProtKB/\S+:(\w+)", x) \
             or re.match(r"UniProt(?:KB)?:(\w+)", x)
        if m:
            uniprot_map[locus] = m.group(1)
            break
print(f"UniProt IDs mapped: {len(uniprot_map)}/{len(cds_df)} CDS")

af_input = cds_df[["locus_tag"]].copy()
af_input["uniprot_id"] = af_input["locus_tag"].map(uniprot_map).fillna("")

af = AlphaFoldExtractor()
t0 = time.perf_counter()
af_feats = af.extract(af_input[["locus_tag", "uniprot_id"]])
af_wall = time.perf_counter() - t0
af_path, af_sha = af.write_cache(af_feats, cache_dir=CACHE_DIR)
n_struct = int(af_feats["af_has_structure"].sum())
print(f"AlphaFold done in {af_wall:.1f}s  parquet={af_path}")
print(f"  rows={len(af_feats)}  structures found={n_struct}")
print(f"  pLDDT mean (structures only): "
      f"{af_feats.loc[af_feats['af_has_structure'] == 1, 'af_plddt_mean'].mean():.1f}")
print(f"  sha256={af_sha}")


In [ ]:
# Cell 7 — MACE-OFF BDE-derived k_cat refinement.
# Skipped if mace-torch wasn't installable (see cell 2). In that case
# we write an empty mace_off_kcat.parquet with the correct columns so
# the FeatureRegistry can register the source as not-populated without
# special casing.
if not MACE_AVAILABLE:
    print("mace-torch unavailable — writing empty mace_off_kcat.parquet "
          "(all genes NaN, mace_has_estimate = 0.0).")
    from cell_sim.features.extractors import MaceOffExtractor
    mace = MaceOffExtractor(model="small", device="cpu")
    import numpy as np
    n = len(cds_df)
    empty_feats = pd.DataFrame(
        {c: [np.nan] * n for c in mace.feature_cols},
    )
    # has_estimate is 0 for every gene in the no-MACE path.
    empty_feats["mace_has_estimate"] = 0.0
    empty_feats.index = pd.Index(cds_df["locus_tag"].tolist(), name="locus_tag")
    mace_wall = 0.0
    mace_path, mace_sha = mace.write_cache(empty_feats, cache_dir=CACHE_DIR)
    print(f"  parquet={mace_path}  sha256={mace_sha}")
else:
    from cell_sim.layer3_reactions.sbml_parser import parse_sbml
    from cell_sim.features.extractors import MaceOffExtractor

    SBML_PATH = Path(
        "cell_sim/data/Minimal_Cell_ComplexFormation/input_data/Syn3A_updated.xml"
    )
    sbml = parse_sbml(SBML_PATH)
    print(f"SBML loaded: {len(sbml.species)} species, "
          f"{len(sbml.reactions)} reactions")

    # Build the per-reaction substrate table. For each gene-associated
    # reaction we emit one row per substrate species. This is a loose
    # bootstrap — a future session can swap in a curated SMILES
    # mapping once we cite it properly.
    mace_rows = []
    for rxn in sbml.reactions.values():
        if not rxn.gene_associations:
            continue
        for gene_id in rxn.gene_associations:
            locus = gene_id.replace("G_MMSYN1_", "JCVISYN3A_")
            for sp_id, _ in rxn.reactants.items():
                mace_rows.append({
                    "locus_tag": locus,
                    "enzyme_name": rxn.short_name,
                    "reaction_class": "metabolic",
                    # SMILES column is left as the species id for the
                    # bootstrap; MACEBackend will return 'mace_bde_failed'
                    # on rows without a real SMILES, which the
                    # extractor handles as a no-estimate row.
                    "substrate_smiles": sp_id,
                })
    mace_input = pd.DataFrame(mace_rows)
    print(f"MACE inputs: {len(mace_input)} (enzyme, substrate) pairs")

    mace = MaceOffExtractor(model="small", device="auto")
    t0 = time.perf_counter()
    mace_feats = mace.extract(mace_input)
    mace_wall = time.perf_counter() - t0
    mace_path, mace_sha = mace.write_cache(mace_feats, cache_dir=CACHE_DIR)
    n_est = int(mace_feats["mace_has_estimate"].sum())
    print(f"MACE-OFF done in {mace_wall:.1f}s  parquet={mace_path}")
    print(f"  rows={len(mace_feats)}  estimates={n_est}")
    print(f"  sha256={mace_sha}")


In [ ]:
# Cell 8 — refresh manifest.json. write_cache() already added
# each parquet, but we re-load and re-verify here so the user
# sees the full manifest state in one place.
from cell_sim.features.cache_manifest import CachedFeatureManifest

MANIFEST_PATH = CACHE_DIR / "manifest.json"
manifest = CachedFeatureManifest.load(MANIFEST_PATH)
print(f"manifest at {MANIFEST_PATH}")
print(f"  tracked sources: {sorted(manifest.sources)}")
for name, entry in sorted(manifest.sources.items()):
    ok = manifest.verify(name, CACHE_DIR / f"{name}.parquet")
    print(f"  {name:20s}  v{entry['version']}  rows={entry['rows']:<6}  "
          f"sha256={entry['sha256'][:12]}...  verified={ok}")


In [ ]:
# Cell 9 — choose how to ship the populated cache back to your
# repo / machine. Pick exactly one OUTPUT_MODE and leave the
# others commented out.

OUTPUT_MODE = "github_pat"   # options: "drive" | "download" | "github_pat"

# --- Path 1: Google Drive mount ---
if OUTPUT_MODE == "drive":
    from google.colab import drive  # type: ignore
    import shutil
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/cell_sim_cache")
    dest.mkdir(parents=True, exist_ok=True)
    for name in ("esm2_650M", "alphafold_db", "mace_off_kcat"):
        src = CACHE_DIR / f"{name}.parquet"
        shutil.copy(src, dest / src.name)
    shutil.copy(MANIFEST_PATH, dest / "manifest.json")
    print(f"Copied parquets + manifest.json to {dest}")
    print("Sync them into cell_sim/features/cache/ on your local "
          "checkout and commit manually.")

# --- Path 2: direct browser download (one prompt per file) ---
elif OUTPUT_MODE == "download":
    from google.colab import files  # type: ignore
    for name in ("esm2_650M", "alphafold_db", "mace_off_kcat"):
        files.download(str(CACHE_DIR / f"{name}.parquet"))
    files.download(str(MANIFEST_PATH))

# --- Path 3: GitHub PAT push (fully automated) ---
elif OUTPUT_MODE == "github_pat":
    import os, subprocess
    pat = os.environ.get("GITHUB_PAT")
    if not pat:
        raise RuntimeError(
            "Set GITHUB_PAT in Colab secrets or "
            "%env GITHUB_PAT=ghp_... before running this cell."
        )

    def run(cmd: list[str], check: bool = True, **kw) -> None:
        res = subprocess.run(cmd, capture_output=True, text=True, **kw)
        if res.stdout.strip():
            print(res.stdout.rstrip())
        if res.stderr.strip():
            print(res.stderr.rstrip())
        if check and res.returncode != 0:
            raise RuntimeError(
                f"{cmd!r} failed with exit {res.returncode}"
            )

    run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
    run(["git", "config", "user.name", "cell-sim-bot"])
    # Force-add the parquets: .gitignore keeps them out by default
    # (they are derivable outputs), but we want this one commit to
    # ship them to origin.
    run(["git", "add", "-f",
         "cell_sim/features/cache/manifest.json",
         "cell_sim/features/cache/esm2_650M.parquet",
         "cell_sim/features/cache/alphafold_db.parquet",
         "cell_sim/features/cache/mace_off_kcat.parquet"])
    run(["git", "status", "--short"])
    run(["git", "commit", "-m",
         "Session 14: populate Tier 1 cache (ESM-2, AlphaFold-DB, MACE-OFF)"])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote,
         "claude/syn3a-whole-cell-simulator-REjHC"])
    print("Push complete.")
else:
    raise ValueError(
        f"unknown OUTPUT_MODE={OUTPUT_MODE!r}; "
        "choose drive | download | github_pat"
    )


In [ ]:
# Cell 10 — final summary. Paste the output back to the review
# thread so Session 15 can verify the SHAs before running the
# downstream detector stack.
from textwrap import dedent
manifest = CachedFeatureManifest.load(MANIFEST_PATH)
lines = ["## Tier-1 cache population summary", ""]
for name, entry in sorted(manifest.sources.items()):
    parquet = CACHE_DIR / f"{name}.parquet"
    df = pd.read_parquet(parquet)
    lines.append(f"### {name}  (v{entry['version']})")
    lines.append(f"- parquet: `{parquet}`")
    lines.append(f"- rows: {entry['rows']}")
    lines.append(f"- sha256: `{entry['sha256']}`")
    lines.append(f"- created_at: {entry['created_at']}")
    lines.append(
        f"- first 3 locus_tags: "
        f"{list(df.index[:3])}"
    )
    sample_cols = list(df.columns[:5])
    lines.append(f"- first 5 columns: {sample_cols}")
    if sample_cols:
        first_values = df.iloc[0][sample_cols].to_dict()
        lines.append(f"- first-row values: {first_values}")
    lines.append("")
wall_lines = [
    f"- ESM-2 wall:      {esm_wall:.1f}s",
    f"- AlphaFold wall:  {af_wall:.1f}s",
    f"- MACE-OFF wall:   {mace_wall:.1f}s",
]
lines.append("### Wall times")
lines.extend(wall_lines)
print("\n".join(lines))
